Cell 1 — Check environment

In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

Cell 2 — Setup project root

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Cell 3 — Load dataset for CNN full training

In [ ]:
import tensorflow as tf
from pathlib import Path

TRAIN_DIR = PROJECT_ROOT / "data/processed/train"
VAL_DIR = PROJECT_ROOT / "data/processed/val"
TEST_DIR = PROJECT_ROOT / "data/processed/test"

IMG_SIZE = (96, 96)
BATCH_SIZE = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True,
    seed=SEED,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)

class_names = train_ds.class_names
num_classes = len(class_names)

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

print("Number of classes:", num_classes)

Cell 4 — Compute class weight

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

all_train_labels = []

for class_dir in sorted(TRAIN_DIR.iterdir(), key=lambda p: int(p.name)):
    if class_dir.is_dir():
        count = len(list(class_dir.glob("*.*")))
        all_train_labels.extend([int(class_dir.name)] * count)

all_train_labels = np.array(all_train_labels)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(all_train_labels),
    y=all_train_labels,
)

class_weight = {
    int(class_id): float(weight)
    for class_id, weight in zip(np.unique(all_train_labels), class_weights_array)
}

print("Number of class weights:", len(class_weight))
print(list(class_weight.items())[:5])

Cell 5 — Build and train CNN full model

In [ ]:
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "results/figures"
REPORTS_DIR = PROJECT_ROOT / "results/reports"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

CNN_FINAL_MODEL_PATH = MODELS_DIR / "cnn_full_final.keras"
CNN_HISTORY_PATH = FIGURES_DIR / "cnn_full_accuracy_loss.png"

data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.05, 0.05),
], name="data_augmentation")

cnn_model = models.Sequential([
    layers.Input(shape=(96, 96, 3)),
    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Dropout(0.30),

    layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Dropout(0.35),

    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.40),
    layers.Dense(num_classes, activation="softmax"),
])

cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

cnn_model.summary()

callbacks = [
    EarlyStopping(
        monitor="val_accuracy",
        patience=7,
        restore_best_weights=True,
    ),
    ModelCheckpoint(
        CNN_FINAL_MODEL_PATH,
        monitor="val_accuracy",
        save_best_only=True,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
    ),
]

EPOCHS = 30

cnn_history = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight,
)

Cell 6 — Plot CNN full training history

In [ ]:
history = cnn_history.history

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history["accuracy"], label="Training Accuracy")
plt.plot(history["val_accuracy"], label="Validation Accuracy")
plt.title("CNN Full Training Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history["loss"], label="Training Loss")
plt.plot(history["val_loss"], label="Validation Loss")
plt.title("CNN Full Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.savefig(CNN_HISTORY_PATH, dpi=300)
plt.show()

print("Saved:", CNN_HISTORY_PATH)
print("Model exists:", CNN_FINAL_MODEL_PATH.exists())

Cell 7 — Evaluate CNN full model

In [ ]:
best_cnn_model = tf.keras.models.load_model(CNN_FINAL_MODEL_PATH)

test_loss, test_accuracy = best_cnn_model.evaluate(test_ds)

print("CNN full test loss:", test_loss)
print("CNN full test accuracy:", test_accuracy)
print("CNN full test accuracy percent:", test_accuracy * 100)

Cell 8 — Save CNN full report and confusion matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

y_true = []
y_pred = []

for images, labels in test_ds:
    predictions = best_cnn_model.predict(images, verbose=0)
    predicted_labels = np.argmax(predictions, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(predicted_labels)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

accuracy = accuracy_score(y_true, y_pred)

macro_precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
macro_recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

weighted_precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
weighted_recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("Accuracy:", accuracy)
print("Macro Precision:", macro_precision)
print("Macro Recall:", macro_recall)
print("Macro-F1:", macro_f1)
print("Weighted Precision:", weighted_precision)
print("Weighted Recall:", weighted_recall)
print("Weighted-F1:", weighted_f1)

report = classification_report(y_true, y_pred, digits=4, zero_division=0)

REPORT_PATH = REPORTS_DIR / "cnn_full_classification_report.txt"

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write("CNN Full Final Classification Report\n")
    f.write("====================================\n\n")
    f.write(f"Test loss: {test_loss:.4f}\n")
    f.write(f"Test accuracy: {test_accuracy:.4f}\n")
    f.write(f"Accuracy: {accuracy:.4f}\n")
    f.write(f"Macro Precision: {macro_precision:.4f}\n")
    f.write(f"Macro Recall: {macro_recall:.4f}\n")
    f.write(f"Macro-F1: {macro_f1:.4f}\n")
    f.write(f"Weighted Precision: {weighted_precision:.4f}\n")
    f.write(f"Weighted Recall: {weighted_recall:.4f}\n")
    f.write(f"Weighted-F1: {weighted_f1:.4f}\n\n")
    f.write(report)

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(12, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(ax=ax, values_format="d", xticks_rotation=90)
plt.title("CNN Full Final Confusion Matrix")
plt.tight_layout()

CM_PATH = FIGURES_DIR / "cnn_full_confusion_matrix.png"
plt.savefig(CM_PATH, dpi=300)
plt.show()

print("Saved report:", REPORT_PATH)
print("Saved confusion matrix:", CM_PATH)